# Inspect Aggregated SNODAS SWE

HRU-level inspection of SNODAS daily snow water equivalent. Companion to `aggregate/snodas.py`.

Source (see `catalog/sources.yml` → `snodas`):

- SNODAS (NSIDC G02158) daily 1 km CONUS SWE in `kg m-2` (≡ mm of water at density 1000), `cell_methods: "time: point"` (instantaneous, not accumulated).

Per the repo's transformation policy (`docs/architecture/transformation-pipeline.md`): the consolidated NC arrives with `_FillValue=-9999` (xarray decodes to NaN at open time), so no pre-aggregation hook is needed and the aggregator uses `stat_method="mean"`. The native `kg m-2` unit passes through unchanged. The `÷ 25.4` rescale to **inches** (PRMS `pkwater_equiv`) is a linear conversion that happens **post-aggregation** — this notebook applies it inline for display, and `targets/swe.py` will apply it for the final calibration target.

## Per-target conventions in this notebook

- The aggregated NC carries `swe` in its **native `kg m-2` (≡ mm)** units. Ocean and out-of-CONUS pixels were `-9999` on disk and decoded to NaN at open time, so HRUs that straddle the CONUS edge come out NaN. NN-fill (if applied) is a target-stage concern, not an aggregator concern.
- This notebook converts to **inches** (÷ 25.4) for the PRMS-units sanity check; the underlying values remain in mm.
- SWE is **instantaneous** (`cell_methods: "time: point"`) — daily snapshots, never accumulated. A monthly aggregate is the mean over time, not a sum.
- Single-source-in-isolation view here: this notebook is one piece of the four-source SWE target (Daymet, SNODAS, ERA5-Land `sd`, Margulis WUS-SR). Cross-source comparison lives in `inspect_aggregated_swe.ipynb` (lands with the target-builder PR).

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr

from _helpers import (
    area_weighted_mean,
    discover_aggregated,
    load_fabric,
    load_project_paths,
    lookup_hrus_by_points,
    nan_hru_count,
    open_year,
    open_year_range,
    plot_hru_choropleth,
    plot_nan_hrus,
    save_figure,
    unit_from_catalog,
)

# Edit me to point at a real project directory:
PROJECT_DIR = Path(
    "/caldera/hovenweep/projects/usgs/water/impd/nhgf/gfv2-spatial-targets"
)

# Set True (and re-run) to populate docs/figures/aggregated/<project>/*.png
import _helpers
_helpers.SAVE_FIGURES = False
_helpers.PROJECT = PROJECT_DIR.name

project_dir, datastore_dir, fabric_cfg = load_project_paths(PROJECT_DIR)
fabric = load_fabric(fabric_cfg)

SOURCE_KEY = "snodas"
VAR = "swe"
# Mid-archive year with a typical CONUS winter pack.
TARGET_YEAR = 2010
TARGET_DAY = "2010-02-15"
TARGET_MONTH = 2  # February
TIME_SERIES_YEARS = range(2003, 2010)
# kg m-2 ≡ mm water; ÷ 25.4 → inches (PRMS pkwater_equiv).
MM_PER_INCH = 25.4

REPRESENTATIVE_POINTS = {
    "Olympic Peninsula (PNW mountains)": (-123.5, 47.8),
    "Northern Rockies (MT/WY)": (-110.5, 45.5),
    "Iowa cropland (Midwest)": (-93.6, 42.0),
    "Southern Appalachians (Eastern forest)": (-83.5, 35.5),
}

print(f"Project: {project_dir}")
print(f"Datastore: {datastore_dir}")
print(f"Fabric: {fabric_cfg['path']} ({len(fabric)} HRUs)")

## Load aggregated dataset

Opens `<project>/data/aggregated/snodas/snodas_<TARGET_YEAR>_agg.nc`. If the year hasn't been aggregated yet, the cell prints a clear skip message and the remaining cells fall through gracefully.

In [ ]:
paths = discover_aggregated(project_dir, SOURCE_KEY)
if paths is None:
    print(
        f"SKIP {SOURCE_KEY}: no aggregated NCs at "
        f"{project_dir}/data/aggregated/{SOURCE_KEY}/. Run "
        f"`pixi run nhf-targets agg snodas --project-dir {project_dir}` first."
    )
    ds = None
else:
    ds = open_year(project_dir, SOURCE_KEY, TARGET_YEAR)
    units = unit_from_catalog(SOURCE_KEY, VAR)
    values_t0 = ds[VAR].isel(time=0).to_pandas()
    print(
        f"SNODAS: vars={list(ds.data_vars)} | "
        f"time={ds.time.values[0]}..{ds.time.values[-1]} | "
        f"HRUs={ds.sizes.get(fabric_cfg['id_col'], 'unknown')} | "
        f"NaN HRUs (t=0): {nan_hru_count(values_t0)} | "
        f"catalog units: {units}"
    )
    display(ds)

## Daily snapshot + monthly mean (mm and inches)

Two HRU choropleths side-by-side: the daily SWE on `TARGET_DAY` and the monthly mean over `TARGET_YEAR-TARGET_MONTH`. Native `kg m-2` (≡ mm) on the left, the PRMS-units (`÷ 25.4` → inches) view on the right. NaN HRUs are light grey — typically a thin band along the CONUS coast where the SNODAS `-9999` ocean mask leaks into HRU footprints.

In [ ]:
if ds is None:
    print("No aggregated SNODAS; skipping panels.")
else:
    da_day_mm = ds[VAR].sel(time=TARGET_DAY, method="nearest")
    monthly_mm = (
        ds[VAR]
        .sel(
            time=slice(
                pd.Timestamp(year=TARGET_YEAR, month=TARGET_MONTH, day=1),
                pd.Timestamp(year=TARGET_YEAR, month=TARGET_MONTH, day=1)
                + pd.offsets.MonthEnd(0),
            )
        )
        .mean("time", skipna=True)
        .to_pandas()
    )
    day_mm = da_day_mm.to_pandas()
    monthly_in = monthly_mm / MM_PER_INCH

    # Use the 95th-percentile of the daily snapshot as a common vmax for the
    # mm panels so the colour scale is dominated by the bulk of the snowy
    # HRUs, not the deepest mountain HRU.
    vmax_mm = float(np.nanpercentile(day_mm.dropna(), 95))

    fig, axes = plt.subplots(2, 2, figsize=(16, 10), squeeze=False)
    plot_hru_choropleth(
        axes[0, 0], fabric, day_mm,
        vmin=0, vmax=vmax_mm, cmap="Blues",
        title=f"Daily SWE\n{TARGET_DAY}",
        units="mm (kg m-2)",
    )
    plot_hru_choropleth(
        axes[0, 1], fabric, monthly_mm,
        vmin=0, vmax=vmax_mm, cmap="Blues",
        title=f"Monthly mean SWE\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="mm (kg m-2)",
    )
    plot_hru_choropleth(
        axes[1, 0], fabric, day_mm / MM_PER_INCH,
        vmin=0, vmax=vmax_mm / MM_PER_INCH, cmap="Blues",
        title=f"Daily SWE (PRMS units)\n{TARGET_DAY}",
        units="inches",
    )
    plot_hru_choropleth(
        axes[1, 1], fabric, monthly_in,
        vmin=0, vmax=vmax_mm / MM_PER_INCH, cmap="Blues",
        title=f"Monthly mean SWE (PRMS units)\n{TARGET_YEAR}-{TARGET_MONTH:02d}",
        units="inches",
    )
    fig.suptitle(
        "SNODAS SWE on HRU fabric — aggregator preserves native kg m-2; PRMS ÷ 25.4 rescale here",
        fontsize=12, y=1.00,
    )
    plt.tight_layout()
    save_figure(fig, "snodas_native_units_map")
    plt.show()

### Reading the panels

- **Top row — daily snapshot vs. monthly mean (mm).** The daily snapshot is dominated by the highest-elevation HRUs (Cascades, Sierras, Rockies, Adirondacks). The monthly mean is a smoothed version: it should preserve the same broad geographic pattern but with somewhat lower mid-month melt-and-redeposit amplitudes. If the monthly map looks visually identical to the daily, suspect a stuck time index.
- **Bottom row — same data in PRMS units (inches).** This is the unit the SWE target ultimately emits. The visual pattern is identical to the top row (linear rescale); only the numeric scale changes.
- **NaN HRUs.** Coastal HRUs whose pixel footprint includes ocean cells (`-9999` on disk) come out NaN by design — see the coverage cell below.
- **Red flags.** A unimodal map with no snow in the Rockies in February is a missed `_FillValue` decode or a units-conversion error somewhere; a wildly compressed colour scale with no spatial structure usually means `mask_and_scale=True` failed and integer `-9999` is dominating the mean.

In [ ]:
if ds is not None:
    fig, ax = plt.subplots(figsize=(10, 5))
    monthly_finite_mm = monthly_mm.dropna()
    ax.hist(monthly_finite_mm, bins=60, histtype="step", linewidth=2, density=True)
    ax.set_xlabel("SWE (mm, monthly mean)")
    ax.set_ylabel("Density")
    ax.set_title(
        f"HRU-level monthly-mean SWE distribution — {TARGET_YEAR}-{TARGET_MONTH:02d}"
    )
    save_figure(fig, "snodas_histogram")
    plt.show()
    print(
        f"monthly mean SWE summary (mm): "
        f"min={monthly_finite_mm.min():.2f}, "
        f"median={monthly_finite_mm.median():.2f}, "
        f"mean={monthly_finite_mm.mean():.2f}, "
        f"95th={np.nanpercentile(monthly_finite_mm, 95):.2f}, "
        f"max={monthly_finite_mm.max():.2f}"
    )

### Reading the SWE histogram

Expected shape for a winter month is **right-skewed with a heavy zero peak**:

- A tall spike at **0 mm** — most HRUs (lower elevation / southern CONUS) carry essentially no snow in any given winter month.
- A long right tail — mountain HRUs reach hundreds of mm of SWE; the deepest Sierra and PNW maritime HRUs can exceed 1000 mm in peak months.
- A near-continuous distribution between, populated by mid-elevation and northern-CONUS HRUs.

Useful red flags:

- **No HRUs above ~200 mm in February.** Either the consolidated NC missed the deep-pack regions, or `-9999` is being treated as data (bringing the mean down).
- **A spike at a non-zero round value** (e.g. a peak at exactly `-9999 / 25.4` mm) means xarray's `mask_and_scale` failed on read.
- **A unimodal Gaussian-looking distribution** in winter would be wrong — SWE is fundamentally bimodal (snowy vs not).

In [ ]:
if ds is not None:
    rep_hrus = lookup_hrus_by_points(fabric, REPRESENTATIVE_POINTS)
    print("Representative HRUs:", rep_hrus)

    ds_range = open_year_range(project_dir, SOURCE_KEY, TIME_SERIES_YEARS)
    try:
        id_dim = fabric_cfg["id_col"]
        swe_full_mm = (
            ds_range[VAR]
            .sel({id_dim: list(rep_hrus.values())})
            .load()
        )
    finally:
        ds_range.close()

    swe_full_in = swe_full_mm / MM_PER_INCH

    fig, axes = plt.subplots(2, 2, figsize=(14, 8), sharex=True)
    for ax, (label, hru_id) in zip(axes.flat, rep_hrus.items()):
        ax.plot(
            swe_full_in.time,
            swe_full_in.sel({id_dim: hru_id}).values,
            linewidth=0.6,
        )
        ax.set_title(f"{label} (HRU {hru_id})")
        ax.set_ylabel("SWE (inches)")
        ax.grid(alpha=0.3)
    fig.suptitle(
        f"SNODAS SWE at representative HRUs — {min(TIME_SERIES_YEARS)}–"
        f"{max(TIME_SERIES_YEARS)} (daily, PRMS units)"
    )
    plt.tight_layout()
    save_figure(fig, "snodas_time_series")
    plt.show()

### Representative-HRU seasonality check

Each panel is a multi-year daily SWE time series at one HRU. Expected signatures (in inches, PRMS units):

- **Olympic Peninsula / Northern Rockies (mountain HRUs)** — Strong winter peaks (Dec–Apr) reaching tens of inches; the deepest maritime snowpacks can exceed 100 in mid-winter. Slow spring melt curve through May–Jun, near zero Jul–Oct. Strong inter-year variability driven by ENSO / PDO.
- **Iowa cropland (Midwest)** — Sharper winter peaks (Jan–Feb) usually under ~10 inches, with rapid early-spring melt and zero pack May–Oct. Years with persistent polar-vortex incursions show extended Feb–Mar persistence.
- **Southern Appalachians (Eastern forest)** — Short snow pulses lasting days to a week or two, small amplitude (< 5 inches typically). The trace should look like a series of spikes rather than a continuous winter pack.

**Red flags.** A flat line at zero in the mountain panels is a sign that the HRU footprint missed the high-elevation pixel band (geometry / projection issue), not that SNODAS lacks snow. A constant non-zero baseline in the southern HRU is a missed fill-value or an integer-mean leak.

In [ ]:
if ds is not None:
    n_nan = nan_hru_count(monthly_mm)
    print(
        f"SNODAS {TARGET_YEAR}-{TARGET_MONTH:02d}: "
        f"{n_nan} NaN HRUs ({100 * n_nan / len(fabric):.2f}%)"
    )
    fig, axes = plt.subplots(1, 1, figsize=(10, 6), squeeze=False)
    plot_nan_hrus(
        axes[0, 0],
        fabric,
        monthly_mm,
        title=(
            f"NaN HRUs (red) — {n_nan} of {len(fabric)} "
            f"(monthly mean, {TARGET_YEAR}-{TARGET_MONTH:02d})"
        ),
    )
    fig.suptitle(
        "Coverage gaps — CONUS edge ocean masks; NN-fill happens at target stage",
        fontsize=12, y=1.02,
    )
    plt.tight_layout()
    save_figure(fig, "snodas_coverage")
    plt.show()

### Coverage-gap interpretation

SNODAS distributes a **CONUS-masked** product: pixels outside the CONUS analysis domain (oceans, Canada, Mexico) are `_FillValue=-9999` on disk and decode to NaN at open time. HRUs whose footprint straddles the CONUS edge therefore aggregate NaN under the area-weighted mean.

Two distinct populations land red:

- **Coast-clipping HRUs.** Long, thin, or peninsular HRUs along the Atlantic, Pacific, and Gulf coasts. The aggregator is honest about geometric partial coverage here — NaN means "a non-trivial fraction of the HRU is outside SNODAS coverage," not "no snow."
- **Border-state HRUs.** A handful of HRUs that touch the Canadian or Mexican border can also come out NaN for the same reason.

**Downstream handling.** The SWE target builder runs **multi-source NaN-aware min/max** across four sources (Daymet, SNODAS, ERA5-Land `sd`, Margulis WUS-SR), so an HRU that's NaN in SNODAS often gets a finite bound from another source. NN-fill (when applied) is a *target-stage post-processing step* on the combined bound, not on the aggregated NC — see `targets/run.py` for the canonical NN-fill pattern.

Expect the NaN fraction in any given winter month to be **well under ~5%** of the CONUS fabric; if it climbs into double digits, suspect a fetch / consolidation regression rather than an aggregator regression.

In [ ]:
if ds is not None:
    # Magnitude sanity check: HRU-area-weighted monthly mean (mm) over the
    # fabric, compared with a typical SNODAS CONUS-mean SWE for the same
    # month. Published CONUS-mean winter peak SNODAS SWE lands in the
    # tens-of-mm range (e.g. February ≈ 20–40 mm depending on year).
    # Memory: feedback_validate_magnitudes — |% diff| > 30% from a published
    # mean is a smoking gun for a missed conversion factor.
    agg_mean_mm = area_weighted_mean(monthly_mm, fabric)
    agg_mean_in = agg_mean_mm / MM_PER_INCH if agg_mean_mm == agg_mean_mm else float("nan")
    print(
        f"SNODAS {TARGET_YEAR}-{TARGET_MONTH:02d} HRU-area-weighted mean: "
        f"{agg_mean_mm:.2f} mm  ({agg_mean_in:.3f} inches)"
    )
    print(
        "Expected order of magnitude for CONUS winter peak: tens of mm "
        "(typical February ≈ 20–40 mm); a result an order of magnitude off "
        "is a smoking gun for a missed _FillValue decode or unit conversion."
    )

## Why this is a single-source diagnostic, not a cross-source comparison

SNODAS is one of four sources for the SWE calibration target. This notebook is checking that the SNODAS-side aggregation is sane on its own; the cross-source view (with Daymet, ERA5-Land `sd`, and Margulis WUS-SR alongside) lives in the to-be-written `inspect_aggregated_swe.ipynb`.

- **No quality gate at aggregate time.** Unlike MOD10C1, SNODAS has no per-pixel CI gate. The only mask is `_FillValue=-9999` (ocean / out-of-CONUS), and xarray decodes that to NaN at open time. The aggregator therefore uses `stat_method="mean"` with no `pre_aggregate_hook`; HRUs with any NaN pixel propagate to NaN at the HRU level (honest geometric partial coverage).
- **Units pass through.** The aggregator preserves native `kg m-2` (≡ mm). The `÷ 25.4` rescale to inches is the notebook's / target's job; CLAUDE.md's transformation policy reserves linear unit conversions for `targets/`.
- **Time is point, not sum.** SWE is `cell_methods: "time: point"`. A monthly aggregate is the mean over daily instantaneous values, not a sum of accumulations. This matters when reading TM 6-B10 §SWE: the report treats SWE as a state variable, not a flux.

**Calibration target implication.** The SWE target (`targets/swe.py`, separate PR) reads this aggregated NC, combines it with the other three SWE sources via NaN-aware min/max, optionally NN-fills the combined bound on HRUs that were NaN everywhere, and emits the final calibration target as PRMS `pkwater_equiv` in inches. What this notebook checks is that the SNODAS-side input to that pipeline is sane on its own.

In [ ]:
if ds is not None:
    ds.close()